# Modelo de Regresión Logística Binaria: Cercanía a Bregman

Este notebook ajusta modelos de regresión logística binaria recursivos para predecir la cercanía a Bregman en Generales y Ballotage.

**Variable dependiente binarizada:**
- Cercano (1): Cercanía ≥ 4 (valores 4 y 5)
- No cercano (0): Cercanía ≤ 3 (valores 1, 2 y 3)

**Proceso:**
1. Cargar datos y binarizar variable dependiente
2. Ajustar modelo inicial con todas las variables
3. Filtrar variables significativas (p < 0.05)
4. Re-ajustar modelo solo con significativas
5. Repetir hasta convergencia (mismas variables significativas)
6. Reportar: AIC, BIC, Deviance explicada, χ², p-valor, Odds Ratios

In [ ]:
# ============================================================
# IMPORTACIONES
# ============================================================

import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================
# CARGAR DATOS
# ============================================================

Ruta_Datos = (
    'C:/Users/Patricio/Documents/Codigo/Python/'
    'Investigacion/Tesis/Data/Bases definitivas/'
)

df_Generales = pd.read_excel(Ruta_Datos + 'Generales.xlsx')
df_Ballotage = pd.read_excel(Ruta_Datos + 'Ballotage.xlsx')

print(f"Generales: {len(df_Generales)} observaciones")
print(f"Ballotage: {len(df_Ballotage)} observaciones")

In [ ]:
# ============================================================
# BINARIZAR VARIABLE DEPENDIENTE
# ============================================================

# Variable original.
Variable_Cercania = 'Cercania_Bregman'

# Crear variable binaria: 1 si cercano (≥4), 0 si no (≤3).
df_Generales['Cercano_Bregman'] = (
    df_Generales[Variable_Cercania] >= 4
).astype(int)

df_Ballotage['Cercano_Bregman'] = (
    df_Ballotage[Variable_Cercania] >= 4
).astype(int)

# Variable dependiente binaria.
Variable_Dependiente = 'Cercano_Bregman'

# Verificar distribución.
print(f"\nDistribución en Generales:")
print(df_Generales[Variable_Dependiente].value_counts())
print(
    f"Proporción cercanos: "
    f"{df_Generales[Variable_Dependiente].mean():.2%}"
)

print(f"\nDistribución en Ballotage:")
print(df_Ballotage[Variable_Dependiente].value_counts())
print(
    f"Proporción cercanos: "
    f"{df_Ballotage[Variable_Dependiente].mean():.2%}"
)

In [ ]:
# ============================================================
# DEFINIR VARIABLES INDEPENDIENTES
# ============================================================

# Variables independientes (excluyendo cercanías a candidatos).
Variables_Independientes = [
    'Conservative_Cluster',
    'Progressive_Cluster',
    'Edad',
    'Genero_Masculino',
    'Genero_Otro',
    'Region_CABA',
    'Region_Centro',
    'Region_Cuyo',
    'Region_Norte',
    'Region_Patagonia',
    'Voto_2019_JL_Espert',
    'Voto_2019_J_Gomez_Centurion',
    'Voto_2019_Mauricio_Macri',
    'Voto_2019_Nicolas_Del_Caño',
    'Voto_2019_Roberto_Lavagna',
    'Categoria_PASO_2023_Left_Wing',
    'Categoria_PASO_2023_Moderate_Right_A',
    'Categoria_PASO_2023_Moderate_Right_B',
    'Categoria_PASO_2023_Right_Wing_Libertarian',
    'Categoria_PASO_2023_Centre',
    'Autopercepcion_Izq_Der',
    'Autopercepcion_Con_Pro',
    'Autopercepcion_Per_Antiper',
    'Influencia_Redes',
    'Red_Social_Facebook',
    'Red_Social_Instagram',
    'Red_Social_Threads',
    'Red_Social_Tiktok',
    'Red_Social_Youtube',
    'Red_Social_Whatsapp',
    'Red_Social_Telegram',
    'Influencia_Prensa',
    'Medios_Prensa_Prensa_Obrera',
    'Medios_Prensa_Diario_Universal',
    'Medios_Prensa_Popular',
    'Medios_Prensa_Izquierda_Diario',
    'Medios_Prensa_Clarin',
    'Medios_Prensa_Perfil',
    'Medios_Prensa_Pagina_12',
    'Medios_Prensa_Infobae',
    'Medios_Prensa_El_Cronista',
    'Medios_Prensa_La_Nacion',
    'Medios_Prensa_Tiempo_Argentino',
    'Indice_Positividad',
    'Indice_Progresismo',
    'Indice_Progresismo_Tiempo',
    'Indice_Conservadurismo',
    'Indice_Conservadurismo_Tiempo'
]

In [ ]:
# ============================================================
# FUNCIÓN: AJUSTAR REGRESIÓN LOGÍSTICA RECURSIVA
# ============================================================

def Ajustar_Regresion_Logistica_Recursiva(
    df,
    Variable_Dependiente,
    Variables_Independientes,
    Nombre_Eleccion
):

    """
    Ajusta modelo de regresión logística recursivamente hasta 
    convergencia.
    
    Parámetros:
    - df: DataFrame con los datos.
    - Variable_Dependiente: nombre de variable dependiente 
      binaria.
    - Variables_Independientes: lista de variables 
      independientes.
    - Nombre_Eleccion: string identificador.
    
    Retorna:
    - Diccionario con modelo final y estadísticos.
    
    """

    print(f"\n{'='*60}")
    print(f"REGRESIÓN LOGÍSTICA RECURSIVA: {Nombre_Eleccion}")
    print(f"{'='*60}\n")
    
    # Preparar datos eliminando NaN.
    Columnas_Necesarias = (
        [Variable_Dependiente] + Variables_Independientes
    )
    
    df_Limpio = df[Columnas_Necesarias].dropna()
    
    print(
        f"Observaciones válidas: {len(df_Limpio)} "
        f"de {len(df)}"
    )
    
    # Verificar que hay variabilidad en la variable dependiente.
    if df_Limpio[Variable_Dependiente].nunique() < 2:
        print(
            f"\n❌ ERROR: Variable dependiente no tiene "
            f"variabilidad"
        )
        return None
    
    # Inicializar variables para iteración.
    Variables_Actuales = Variables_Independientes.copy()
    Iteracion = 0
    Variables_Anteriores = []
    
    while True:
        Iteracion += 1
        print(f"\n{'─'*60}")
        print(
            f"Iteración {Iteracion}: "
            f"{len(Variables_Actuales)} variables"
        )
        print(f"{'─'*60}")
        
        # Preparar X e y.
        X = df_Limpio[Variables_Actuales].copy()
        y = df_Limpio[Variable_Dependiente]
        
        # Convertir booleanos a int.
        for col in X.columns:
            if X[col].dtype == bool:
                X[col] = X[col].astype(int)
        
        # Agregar constante.
        X_Con_Constante = sm.add_constant(X)
        
        # Ajustar modelo logístico.
        Modelo = sm.GLM(
            y,
            X_Con_Constante,
            family=sm.families.Binomial()
        ).fit()
        
        # Extraer p-valores.
        P_Valores = Modelo.pvalues
        
        # Filtrar variables significativas.
        Variables_Significativas = [
            var for var in Variables_Actuales
            if P_Valores[var] < 0.05
        ]
        
        print(
            f"Variables significativas (p<0.05): "
            f"{len(Variables_Significativas)}"
        )
        
        # Verificar convergencia.
        if (
            set(Variables_Significativas) == 
            set(Variables_Anteriores)
        ):
            print("\n✓ Convergencia alcanzada")
            break
        
        if len(Variables_Significativas) == 0:
            print(
                "\n⚠ No hay variables significativas. "
                "Deteniendo."
            )
            break
        
        # Actualizar para siguiente iteración.
        Variables_Anteriores = Variables_Actuales.copy()
        Variables_Actuales = Variables_Significativas.copy()
    
    # Modelo final.
    print(f"\n{'='*60}")
    print("MODELO FINAL")
    print(f"{'='*60}\n")
    
    # Calcular estadísticos finales.
    AIC = Modelo.aic
    BIC = Modelo.bic
    
    # Calcular Deviance explicada (Pseudo R² McFadden).
    Deviance_Nulo = Modelo.null_deviance
    Deviance_Residual = Modelo.deviance
    Pseudo_R2 = 1 - (Deviance_Residual / Deviance_Nulo)
    
    # Calcular estadístico chi-cuadrado.
    N = len(df_Limpio)
    K = len(Variables_Actuales)
    
    Chi2_Estadistico = Deviance_Nulo - Deviance_Residual
    P_Valor_Chi2 = 1 - stats.chi2.cdf(Chi2_Estadistico, K)
    
    # Imprimir resultados.
    print(f"N observaciones: {N}")
    print(f"Variables en modelo final: {K}")
    print(f"\nAIC: {AIC:.2f}")
    print(f"BIC: {BIC:.2f}")
    print(f"Deviance nulo: {Deviance_Nulo:.2f}")
    print(f"Deviance residual: {Deviance_Residual:.2f}")
    print(
        f"Deviance explicada (Pseudo R² McFadden): "
        f"{Pseudo_R2:.4f}"
    )
    print(f"χ² estadístico: {Chi2_Estadistico:.4f}")
    print(f"P-valor (χ²): {P_Valor_Chi2:.6e}")
    
    print(f"\n{'─'*60}")
    print(
        "VARIABLES SIGNIFICATIVAS FINALES "
        "(con Odds Ratios):"
    )
    print(f"{'─'*60}\n")
    
    # Calcular Odds Ratios y ordenar por p-valor.
    Resultados_Vars = []
    
    for var in Variables_Actuales:
        Coef = Modelo.params[var]
        P_Val = Modelo.pvalues[var]
        IC = Modelo.conf_int().loc[var]
        
        # Calcular Odds Ratio y sus IC.
        OR = np.exp(Coef)
        OR_IC_Inf = np.exp(IC[0])
        OR_IC_Sup = np.exp(IC[1])
        
        Resultados_Vars.append({
            'Variable': var,
            'Coeficiente': Coef,
            'OR': OR,
            'P_Valor': P_Val,
            'OR_IC_Inferior': OR_IC_Inf,
            'OR_IC_Superior': OR_IC_Sup
        })
    
    df_Resultados = pd.DataFrame(Resultados_Vars)
    df_Resultados = df_Resultados.sort_values('P_Valor')
    
    for idx, fila in df_Resultados.iterrows():
        print(
            f"{fila['Variable']:<45} "
            f"OR={fila['OR']:7.3f}  "
            f"p={fila['P_Valor']:.6e}  "
            f"IC 95%=[{fila['OR_IC_Inferior']:6.3f}, "
            f"{fila['OR_IC_Superior']:6.3f}]"
        )
    
    # Retornar diccionario con resultados.
    return {
        'Modelo': Modelo,
        'Eleccion': Nombre_Eleccion,
        'N_Observaciones': N,
        'N_Variables': K,
        'AIC': AIC,
        'BIC': BIC,
        'Deviance_Nulo': Deviance_Nulo,
        'Deviance_Residual': Deviance_Residual,
        'Pseudo_R2_McFadden': Pseudo_R2,
        'Chi2_Estadistico': Chi2_Estadistico,
        'P_Valor_Chi2': P_Valor_Chi2,
        'Variables_Significativas': Variables_Actuales,
        'Resultados_Variables': df_Resultados,
        'N_Iteraciones': Iteracion
    }

In [ ]:
# ============================================================
# AJUSTAR MODELO: GENERALES
# ============================================================

Resultado_Generales = Ajustar_Regresion_Logistica_Recursiva(
    df=df_Generales,
    Variable_Dependiente=Variable_Dependiente,
    Variables_Independientes=Variables_Independientes,
    Nombre_Eleccion='GENERALES'
)

In [ ]:
# ============================================================
# AJUSTAR MODELO: BALLOTAGE
# ============================================================

# NOTA: Bregman solo participó en Generales.
# No hay modelo para Ballotage.

Resultado_Ballotage = None
print(f"
Bregman solo participó en Generales")

In [ ]:
# ============================================================
# RESUMEN COMPARATIVO
# ============================================================

print(f"\n\n{'='*70}")
print(f"RESUMEN COMPARATIVO: Cercanía a Bregman")
print(f"{'='*70}\n")

if Resultado_Ballotage is not None:
    Resumen_Comparativo = pd.DataFrame([
    {
        'Elección': 'Generales',
        'N': Resultado_Generales['N_Observaciones'],
        'Variables': Resultado_Generales['N_Variables'],
        'AIC': Resultado_Generales['AIC'],
        'BIC': Resultado_Generales['BIC'],
        'Pseudo R²': Resultado_Generales['Pseudo_R2_McFadden'],
        'χ²': Resultado_Generales['Chi2_Estadistico'],
        'p-valor': Resultado_Generales['P_Valor_Chi2'],
        'Iteraciones': Resultado_Generales['N_Iteraciones']
    },
    {
        'Elección': 'Ballotage',
        'N': Resultado_Ballotage['N_Observaciones'],
        'Variables': Resultado_Ballotage['N_Variables'],
        'AIC': Resultado_Ballotage['AIC'],
        'BIC': Resultado_Ballotage['BIC'],
        'Pseudo R²': Resultado_Ballotage['Pseudo_R2_McFadden'],
        'χ²': Resultado_Ballotage['Chi2_Estadistico'],
        'p-valor': Resultado_Ballotage['P_Valor_Chi2'],
        'Iteraciones': Resultado_Ballotage['N_Iteraciones']
    }
])

    print(Resumen_Comparativo.to_string(index=False))
else:
    print(f"
Solo Generales disponible para {config['candidato']}:")
    print(f"  N observaciones: {Resultado_Generales['N_Observaciones']}")
    print(f"  Variables: {Resultado_Generales['N_Variables']}")
    print(f"  AIC: {Resultado_Generales['AIC']:.2f}")
    print(f"  BIC: {Resultado_Generales['BIC']:.2f}")
    print(f"  Pseudo R²: {Resultado_Generales['Pseudo_R2_McFadden']:.4f}")
    print(f"  χ²: {Resultado_Generales['Chi2_Estadistico']:.4f}")
    print(f"  p-valor: {Resultado_Generales['P_Valor_Chi2']:.6e}")

print(f"\n{'─'*70}")
print("Variables significativas en GENERALES:")
print(f"{'─'*70}")
for var in Resultado_Generales['Variables_Significativas']:
    print(f"  • {var}")

print(f"\n{'─'*70}")
print("Variables significativas en BALLOTAGE:")
print(f"{'─'*70}")
for var in Resultado_Ballotage['Variables_Significativas']:
    print(f"  • {var}")